In [24]:
import os
import re
import nltk
from nltk import Tree

import spacy
#spacy.cli.download("en_core_web_sm")
# nlp = spacy.load("en_core_web_sm") 

import pandas as pd
import numpy as np

import textdescriptives as td

from dataset_evaluation.utils import add_column
from dataset_evaluation.evaluation_framework import EvaluationFramework

import folia.main as folia
from pathlib import Path
from collections import defaultdict, Counter
from tqdm.notebook import tqdm

import matplotlib.pyplot as plt
from datasets import load_dataset

import pylangacq
import ast
from matplotlib.lines import Line2D

from vendi_score import text_utils

from diversity import (
	compression_ratio
)

import seaborn as sns
from matplotlib.patches import Patch

plt.style.use("seaborn-v0_8-whitegrid")

## Load datasets

### Baseline

In [25]:
llama = pd.read_csv('/Users/sabijn/Documents/PhD/code/storylm_p1_data/results/results_remote_V2/model_test_llama_5.csv')
llama.rename(columns={"stories": "story"}, inplace=True)

gemma = pd.read_csv('/Users/sabijn/Documents/PhD/code/storylm_p1_data/results/results_remote_V2/model_test_gemma_5.csv')
gemma.rename(columns={"stories": "story"}, inplace=True)

phi = pd.read_csv('/Users/sabijn/Documents/PhD/code/storylm_p1_data/results/results_remote_V2/model_test_phi_5.csv')
phi.rename(columns={"stories": "story"}, inplace=True)

### ChiSCor

In [26]:
path_name = '/Users/sabijn/Documents/PhD/Datasets/chisor_dataset_all/ChiSCor_CoNLL_paper/csv/ChiSCor_master_df_password/ChiSCor_master_df.csv'
df_chiscor = pd.read_csv(path_name, index_col=0)
df_chiscor = df_chiscor.rename(columns={'story_raw': 'story'})

### Reference corpora

In [27]:
ref_standard_dep = Path('datasets/BasiScript/BS_dep_lexicon.csv')
ref_standard_uni = Path('datasets/BasiScript/BS_unigram_lexicon.csv')
ref_standard_bi = Path('datasets/BasiScript/BS_bigram_lexicon.csv')

In [28]:
ref_spoken_b_csv = Path('/Users/sabijn/Documents/PhD/code/storylm_p1_data/datasets/CGN/CGN_pos_bigram.csv')
ref_spoken_u_csv  = Path('/Users/sabijn/Documents/PhD/code/storylm_p1_data/datasets/CGN/CGN_pos_unigram.csv')
ref_spoken_t_csv = Path('/Users/sabijn/Documents/PhD/code/storylm_p1_data/datasets/CGN/CGN_pos_trigram.csv')

In [29]:
all_datasets = {'llama': llama, 'gemma': gemma, 'phi': phi, 'chiscor': df_chiscor}

## Story quality

Overview used metrics:
- Coherence
	- Local contextuality
- Grammaticality
	- Grammaticality
- Surprise
	- Creative perplexity
- Diversity
	- Lexical diversity: self-bleu
	- lexical diversity: moving mtld
- Complexity
	- Lexical complexity: unique words
	- Lexical complexity: Average word length
	- Syntactic complexity: avg. components
	- Syntactic complexity: dependency distance
	- Syntactic complexity: syntactic tree depth

In [30]:
eval_f = EvaluationFramework(language='nl',
							  pos_unigram=ref_spoken_u_csv, 
							  pos_bigram=ref_spoken_b_csv,
							  pos_trigram=ref_spoken_t_csv,
							  ref_unigram=ref_standard_uni,
							  ref_bigram=ref_standard_bi,
							  ref_ling_constrained=ref_standard_dep,
							  embedding_model='jegormeister/bert-base-dutch-cased')

In [31]:
# Surprise: creative perplexity
eval_f.add_pipe('creative_perplexity_dep')
# Local contextuality
eval_f.add_pipe("local_contextuality")
# Grammaticality
eval_f.add_pipe('grammaticality')
# Diversity (lexical) self-bleu
eval_f.add_pipe('self-bleu')
# Diversity (lexical) moving mtld
eval_f.add_pipe('lexical_diversity')
# Complexity (lexical) unique words
eval_f.add_pipe('unique-words')
# Complexity (lexical) average word length
eval_f.add_pipe('avg-word-length')
# Complexity (syntactic)  average components
eval_f.add_pipe('average_components')
# Complexity (syntactic) dependency distance
eval_f.add_pipe('dependency_distance')
# Complexity (syntactic) syntactic tree depth
eval_f.add_pipe('syntactic_depth')
# Words before root
eval_f.add_pipe('wbr_average')

In [32]:
def load_or_run_eval(eval_f, dataset, column, path_name, *, run=False):
	if Path(path_name).exists() and not run:
		df = pd.read_csv(path_name)
		if 'lexical_diversity' in df.columns:
			df['lexical_diversity'] = df['lexical_diversity'].apply(ast.literal_eval)
			
		return df

	dataset = eval_f.run_pipeline_on_df(dataset, column)
	dataset.to_csv(path_name, index=False)
	return dataset

In [33]:
temp = all_datasets
for name, data in temp.items():
	print(f"Current method: {name}")
	generated_eval_results = load_or_run_eval(eval_f, data, 'story', f'results/metric_results_without_newlines/eval_results_{name}.csv')
	all_datasets[name] = generated_eval_results

Current method: llama
Current method: gemma
Current method: phi
Current method: chiscor


### Plotting story quality

In [34]:
nice_name = {'creative_perplexity_dep': 'Creative perplexity',
			 'local_contextuality': 'Local contextuality',
			 'grammaticality': 'Grammaticality',
			 'self-bleu': 'Self-bleu',
			 'lexical_diversity': 'Moving MTLD',
			 'unique-words': 'Size of vocabulary',
			 'avg-word-length': 'Average word length',
			 'average_components': 'Average components',
			 'dependency_distance': 'Dependency distance',
			 'syntactic_depth': 'Syntactic depth'
			 }

In [35]:
def get_value_from_dictionary(x, key):
	new = []

	for item in x:
		new.append(item[key])
	return new

In [36]:
for name, data in all_datasets.items():
	print(data.columns)

Index(['Unnamed: 0', 'system_prompt', 'user_prompt', 'story',
       'creative_perplexity_dep', 'local_contextuality', 'grammaticality',
       'self-bleu', 'lexical_diversity', 'unique-words', 'avg-word-length',
       'average_components', 'dependency_distance', 'syntactic_depth',
       'wbr_average'],
      dtype='object')
Index(['Unnamed: 0', 'system_prompt', 'user_prompt', 'story',
       'creative_perplexity_dep', 'local_contextuality', 'grammaticality',
       'self-bleu', 'lexical_diversity', 'unique-words', 'avg-word-length',
       'average_components', 'dependency_distance', 'syntactic_depth',
       'wbr_average'],
      dtype='object')
Index(['Unnamed: 0', 'system_prompt', 'user_prompt', 'story',
       'creative_perplexity_dep', 'local_contextuality', 'grammaticality',
       'self-bleu', 'lexical_diversity', 'unique-words', 'avg-word-length',
       'average_components', 'dependency_distance', 'syntactic_depth',
       'wbr_average'],
      dtype='object')
Index(['id', 

In [37]:
metrics = ['creative_perplexity_dep', 'local_contextuality', 'grammaticality', 'self-bleu', 'lexical_diversity', 'unique-words', 'avg-word-length', 'average_components', 'dependency_distance', 'syntactic_depth']
story_metrics_means = {}

for name in all_datasets.keys():
	story_metrics_means[name] = defaultdict(float)

for name, data in all_datasets.items():
	for column in data.columns:
		if column in metrics:
			try:
				story_metrics_means[name][column] = data[column].mean()
			except:
				x = get_value_from_dictionary(data[column], 'moving_mtld')
				story_metrics_means[name][column] = np.mean(np.array(x))
story_metrics_means


{'llama': defaultdict(float,
             {'creative_perplexity_dep': 282.63089657913235,
              'local_contextuality': 0.3408936801207672,
              'grammaticality': -2.1850312460549275,
              'self-bleu': 0.1454674059349213,
              'lexical_diversity': 53.41621723305934,
              'unique-words': 87.0,
              'avg-word-length': 3.9552538948178944,
              'average_components': 0.2384509693205345,
              'dependency_distance': 2.8250222187204534,
              'syntactic_depth': 2.6078016186711834}),
 'gemma': defaultdict(float,
             {'creative_perplexity_dep': 1042.2353025557736,
              'local_contextuality': 0.34943911963341623,
              'grammaticality': -2.3007927854626713,
              'self-bleu': 0.12470681077479855,
              'lexical_diversity': 62.10556649024719,
              'unique-words': 110.6,
              'avg-word-length': 3.892442803967735,
              'average_components': 0.104464285714

In [38]:
metrics = ['creative_perplexity_dep', 'local_contextuality', 'grammaticality', 'self-bleu', 'lexical_diversity', 'unique-words', 'avg-word-length', 'average_components', 'dependency_distance', 'syntactic_depth']
story_metrics_std = {}

for name in all_datasets.keys():
	story_metrics_std[name] = defaultdict(float)

for name, data in all_datasets.items():
	for column in data.columns:
		if column in metrics:
			try:
				story_metrics_std[name][column] = data[column].std()
			except:
				x = get_value_from_dictionary(data[column], 'moving_mtld')
				story_metrics_std[name][column] = np.std(np.array(x))
story_metrics_std



{'llama': defaultdict(float,
             {'creative_perplexity_dep': 186.31689479267786,
              'local_contextuality': 0.043935689531369546,
              'grammaticality': 0.11614506860799449,
              'self-bleu': 0.02777532242630583,
              'lexical_diversity': 10.077704141187235,
              'unique-words': 13.619838471876236,
              'avg-word-length': 0.057793603207173916,
              'average_components': 0.11530958387791318,
              'dependency_distance': 0.25323130864227245,
              'syntactic_depth': 0.25621555713936256}),
 'gemma': defaultdict(float,
             {'creative_perplexity_dep': 830.3882738361483,
              'local_contextuality': 0.04930234203182364,
              'grammaticality': 0.028925984858312125,
              'self-bleu': 0.036123840557087006,
              'lexical_diversity': 12.461125365250158,
              'unique-words': 8.44393273303382,
              'avg-word-length': 0.10290534687049556,
            

In [39]:
df_story_means = pd.DataFrame.from_dict(story_metrics_means, orient="index")
df_story_means

,creative_perplexity_dep,local_contextuality,grammaticality,self-bleu,lexical_diversity,unique-words,avg-word-length,average_components,dependency_distance,syntactic_depth
llama,282.630897,0.340894,-2.185031,0.145467,53.416217,87.000000,3.955254,0.238451,2.825022,2.607802
gemma,1042.235303,0.349439,-2.300793,0.124707,62.105566,110.600000,3.892443,0.104464,2.727591,2.005060
phi,2067.033995,0.338581,-2.242068,0.091823,64.535095,145.800000,4.794179,0.359574,3.521695,3.857460
chiscor,695.800719,0.356750,-2.151842,0.160073,30.334225,58.499192,3.699656,0.277128,2.656539,2.365181
